In [51]:
#created on 17/04/2026 by James McLoughlin

In [23]:
from pathlib import Path
import rasterio as rio
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import shape
from rasterio import features
from rasterio.features import shapes
from rasterio.features import sieve


import rasterio.merge
from rasterio.merge import merge


#from rasterio.warp import calculate_default_transform, reproject, Resampling
#from rasterio.io import MemoryFile
#from rasterio.plot import show
#import matplotlib.pyplot as plt


#from rasterio.mask import mask
#from scipy.ndimage import median_filter
#from shapely.geometry import Point
#import os

In [24]:
#base = Path("C:/RS_GIS/EGM722/Assignment/great_salt_lake/USGS_data/Unzipped/two_years")
base = Path("C:/RS_GIS/EGM722/Assignment/Grand_Canyon/USGS_data/Unzipped")
#base = Path("C:/RS_GIS/EGM722/Assignment/Columbia_Icefield/USGS_data/Unzipped")
river_ref = "C:/RS_GIS/EGM722/Assignment/Grand_Canyon/shaprfiles/river.shp"

raw_data = base/"raw_data"
NDI_data = base/"Processed_data"/"NDI"
NDI_data.mkdir(parents=True, exist_ok=True)
mosaic_data = base/"Processed_data"/"Mosaic"
mosaic_data.mkdir(parents=True, exist_ok=True)
mask_data = base/"Processed_data"/"Mask"
mask_data.mkdir(parents=True, exist_ok=True)
polygon_data = base/"Processed_data"/"Polygon"
polygon_data.mkdir(parents=True, exist_ok=True)

NDI_choice = {
    "NDVI": 1,
    "NDWI": 0,
    "NDSI": 1,
}

In [25]:
# Dictionaries describing the bands of the landsat satellites and  
satbands = {
    "LT05":{"B1":"BLUE", "B2":"GREEN", "B3":"RED", "B4":"NIR", "B5":"SWIR1", "B7":"SWIR2"},
    "LT07":{"B1":"BLUE", "B2":"GREEN", "B3":"RED", "B4":"NIR", "B5":"SWIR1", "B6":"TIR", "B7":"SWIR2", "B8":"PAN" },
    "LC08":{"B1":"COAST/AERO", "B2":"BLUE", "B3":"GREEN", "B4":"RED", "B5":"NIR", "B6":"SWIR1", "B7":"SWIR2", "B8":"PAN", "B9":"CIRRUS", "B10":"TIR1", "B11":"TIR2"},
    "LC09":{"B1":"COAST/AERO", "B2":"BLUE", "B3":"GREEN", "B4":"RED", "B5":"NIR", "B6":"SWIR1", "B7":"SWIR2"}
}

index_to_bands = {
    "NDVI": ["RED", "NIR"],
    "NDWI": ["GREEN", "NIR"],
    "NDSI": ["GREEN", "SWIR1"]
}

In [26]:
#Function to calculate NDI
def NDI_computation(yr, pathrow, NDI):
    colr1 = index_to_bands[NDI][0]
    colr2 = index_to_bands[NDI][1]
    path1 = df[(df["colour"] == colr1) & (df["year"] == yr) & (df["path_row"] == pathrow)]["path"].iloc[0]
    path2 = df[(df["colour"] == colr2) & (df["year"] == yr) & (df["path_row"] == pathrow)]["path"].iloc[0]
    
    with rio.open(path1) as src1, rio.open(path2) as src2:
        tile1 = src1.read(1).astype("float32")
        tile2 = src2.read(1).astype("float32")
        with np.errstate(invalid='ignore'):
            ndi_img = (tile1 - tile2) / (tile1+tile2)
        ndi_img[np.isnan(ndi_img)] = -9999 
        ndi_out = NDI_data / f"{yr}_{pathrow}_{NDI}.tif"
        out_meta = src1.meta.copy()
        out_meta.update(dtype = "float32", nodata = -9999)

    with rio.open(ndi_out, "w", **out_meta) as dst:
        dst.write(ndi_img,1)

In [27]:
### function for making mask, binarising it, and converting to polygons
def mask_from_raster(year, pathrow, NDI):#, river_ref_path):
    poly_list = []
    file = NDI_data/f"{year}_{pathrow}_{NDI}.tif"
    with rio.open(file) as src:
        non_binary = src.read(1)
        mask = np.where(non_binary > -0.1 , 255 ,0).astype("uint8")
        cleaned_mask = sieve(mask,50)
        out_meta = src.meta.copy()
        out_meta.update(dtype = "uint8", nodata = 0)
        mask_out = mask_data / f"{year}_{pathrow}_{NDI}.tif"
        
        with rio.open(mask_out, "w", **out_meta) as dst:
            dst.write(cleaned_mask,1)

    file = mask_data / f"{year}_{pathrow}_{NDI}.tif"
    with rio.open(file) as src:
        tile = src.read(1).astype("int16")
        out_meta = src.meta.copy()
        mask_shapes = features.shapes(
            tile,
            mask = tile > 0,
            transform = out_meta['transform']
        )
    # The corrected list comprehension
    polygons = [
    {'geometry':shape(s),'properties':{'id': i}}
     for i, (s,v) in enumerate(mask_shapes)
    ]
              
    gdf = gpd.GeoDataFrame.from_features(polygons, crs=out_meta['crs'])
    return gdf

In [28]:
#Building the dataframe of raw data files
records = []
for folder in raw_data.iterdir():
    if folder.is_dir():
        for file in folder.glob("*_SR_B*.TIF"):
            parts = file.name.split("_")
            sat = parts[0]
            band = parts[-1].replace(".TIF", "")
            records.append({
                "folder": folder.name,
                "file": file.name,
                "path": file,
                "year": parts[3][:4],
                "satellite": sat,
                "path_row": parts[2],
                "band": band,
                "colour": satbands[sat][band]
            })
df = pd.DataFrame(records)

In [40]:
#function for isolationg and joing the river polygons
def merging_polygons(poly_gdf, ref_poly):
    combined_gdf = pd.concat(poly_gdf)
    
    #Apply buffer to bridge gaps between tiles
    combined_gdf['geometry'] = combined_gdf.geometry.buffer(100) 
    
    #Weld small localised pieces together
    merged = combined_gdf.union_all()
    
    # Searate main bodies into distinct polygons
    main_polygons = gpd.GeoSeries([merged], crs="EPSG:26912").explode(index_parts=False)
    main_polygons_gdf = gpd.GeoDataFrame(geometry=main_polygons)
   
    # Keep only polygons that touch river reference
    # ensures small 'ilsnad' polyons in the river path are not omitted.
    ref_geom = gpd.read_file(ref_poly).to_crs("EPSG:26912").union_all()
    referenced_poly = main_polygons_gdf[main_polygons_gdf.intersects(ref_geom)]
   
    # 5. Shrink it back (undo the buffer) to get original river width
    referenced_poly['geometry'] = referenced_poly.geometry.buffer(-100)

    poly_out = polygon_data / "Grand_Canyon_River_Master.gpkg"
    referenced_poly.to_file(poly_out, driver="GPKG")
    print("Master river system saved")  

In [41]:
all_polygons = []
for yr in df["year"].unique():
    for pr in df["path_row"].unique():
        NDI_computation(yr, pr, 'NDWI')
        mask_segment = mask_from_raster(yr, pr, 'NDWI')
        all_polygons.append(mask_segment.to_crs("EPSG:26912"))

merging_polygons(all_polygons, river_ref)

# if all_candidates:
#     combined_gdf = pd.concat(all_candidates)
    
#     #Apply buffer to bridge gaps between tiles
#     combined_gdf['geometry'] = combined_gdf.geometry.buffer(100) 
    
#     #Weld pieces together
#     merged = combined_gdf.union_all()
    
#     # Explode into distinct polygons
#     final_pieces = gpd.GeoSeries([merged], crs="EPSG:26912").explode(index_parts=False)
#     final_gdf = gpd.GeoDataFrame(geometry=final_pieces)
   
#     # Keep only polygons that touch river reference
#     # ensures small 'ilsnad' polyons in the river path are not omitted.
#     river_ref_geom = gpd.read_file(river_ref).to_crs("EPSG:26912").union_all()
#     the_river_system = final_gdf[final_gdf.intersects(river_ref_geom)]
   
#     # 5. Shrink it back (undo the buffer) to get original river width
#     the_river_system['geometry'] = the_river_system.geometry.buffer(-100)

#     final_poly = polygon_data / "Grand_Canyon_River_Master.gpkg"
#     the_river_system.to_file(final_poly, driver="GPKG")
#     print("Master river system saved")       
 

Master river system saved


In [18]:
# ### working backup just in case
# ### function for making mask, converting to polgond and then selcted the largest polgona s well as polgons touching the ref shapefile
# def mask_from_raster(year, pathrow, NDI, ref_shape):#, river_ref_path):
#     poly_list = []
#     file = NDI_data/f"{year}_{pathrow}_{NDI}.tif"
#     with rio.open(file) as src:
#         non_binary = src.read(1)
#         mask = np.where(non_binary > -0.1 , 255 ,0).astype("uint8")
#         cleaned_mask = sieve(mask,50)
#         out_meta = src.meta.copy()
#         out_meta.update(dtype = "uint8", nodata = 0)
#         mask_out = mask_data / f"{year}_{pathrow}_{NDI}.tif"
        
#         with rio.open(mask_out, "w", **out_meta) as dst:
#             dst.write(cleaned_mask,1)

#     file = mask_data / f"{year}_{pathrow}_{NDI}.tif"
#     with rio.open(file) as src:
#         tile = src.read(1).astype("int16")
#         out_meta = src.meta.copy()
#         mask_shapes = features.shapes(
#             tile,
#             mask = tile > 0,
#             transform = out_meta['transform']
#         )
#     # The corrected list comprehension
#     polygons = [
#     {'geometry':shape(s),'properties':{'id': i}}
#      for i, (s,v) in enumerate(mask_shapes)
#     ]
              
#     gdf = gpd.GeoDataFrame.from_features(polygons, crs=out_meta['crs'])
#     # return gdf

#     if not gdf.empty:
#         #return gdf[gdf.geometry.area > 500]
#         gdf["area"] = gdf.geometry.area 
#         largest_polygon = gdf.loc[[gdf["area"].idxmax()]]
        
#         river_ref = gpd.read_file(ref_shape).to_crs(gdf.crs)
#         river_geom = river_ref.union_all()
#         selected_river = gdf[gdf.intersects(river_geom)]

#         # Combine candidates for THIS specific tile
#         candidates = pd.concat([largest_polygon, selected_river]).drop_duplicates()

#         # Return the candidates and the CRS so we can keep track
#         return candidates
#     return None

In [19]:
### working backup just in case
### works with prefilter in mask making fuction
# #############
# if all_candidates:
#     combined_gdf = pd.concat(all_candidates)
    
#     # --- DIAGNOSIS 1: Input Check ---
#     print(f"DEBUG: Total input polygons from all tiles: {len(combined_gdf)}")
    
#     # 1. Apply a small buffer to bridge pixel-sized gaps between tiles
#     combined_gdf['geometry'] = combined_gdf.geometry.buffer(100) 
    
#     # 2. Weld all pieces together
#     merged = combined_gdf.union_all()
    
#     # 3. Explode into distinct islands (The river vs the lake)
#     final_pieces = gpd.GeoSeries([merged], crs="EPSG:26912").explode(index_parts=False)
#     final_gdf = gpd.GeoDataFrame(geometry=final_pieces)

#     # --- DIAGNOSIS 2: Post-Weld Check ---
#     # If this number is '1', the lake and river have physically merged into one object.
#     print(f"DEBUG: Distinct islands after 100m buffer: {len(final_gdf)}")

    
#     # 4. FINAL FILTER: Keep only the islands that touch the river reference
#     # This deletes the lake even if it's huge, because it doesn't touch the line
#     river_ref_geom = gpd.read_file(river_ref).to_crs("EPSG:26912").union_all()
#     the_river_system = final_gdf[final_gdf.intersects(river_ref_geom)]

#     # --- DIAGNOSIS 3: Post-Filter Check ---
#     print(f"DEBUG: Islands remaining after River Reference filter: {len(the_river_system)}")
    
#     # 5. Shrink it back (undo the buffer) to get original river width
#     the_river_system['geometry'] = the_river_system.geometry.buffer(-100)
    
#     # 6. If multiple pieces still exist, take the largest of the survivors
#     if not the_river_system.empty:
#         the_river_system = the_river_system.loc[[the_river_system.geometry.area.idxmax()]]
        
#         final_poly = polygon_data / "Grand_Canyon_River_Master2.gpkg"
#         the_river_system.to_file(final_poly, driver="GPKG")
#         print("Master river system saved, lake removed.")

# #######################


DEBUG: Total input polygons from all tiles: 17
DEBUG: Distinct islands after 100m buffer: 2
Master river system saved, lake removed.


In [19]:
### Unsure if working
# river_ref = "C:/RS_GIS/EGM722/Assignment/Grand_Canyon/shaprfiles/river.shp"
# all_candidates = []
# for yr in df["year"].unique():
#     for pr in df["path_row"].unique():
#         NDI_computation(yr, pr, 'NDWI')
#         mask_segment = mask_from_raster(yr, pr, 'NDWI', river_ref)
#         all_candidates.append(mask_segment.to_crs("EPSG:26912"))


# if all_candidates:
#     combined_gdf = pd.concat(all_candidates)
    
#     # --- DIAGNOSIS 1: Input Check ---
#     print(f"DEBUG: Total input polygons from all tiles: {len(combined_gdf)}")
    
#     # 1. Apply a small buffer to bridge pixel-sized gaps between tiles
#     combined_gdf['geometry'] = combined_gdf.geometry.buffer(100) 
    
#     # 2. Weld all pieces together
#     merged = combined_gdf.union_all()
    
#     # 3. Explode into distinct islands (The river vs the lake)
#     final_pieces = gpd.GeoSeries([merged], crs="EPSG:26912").explode(index_parts=False)
#     final_gdf = gpd.GeoDataFrame(geometry=final_pieces)

#     # --- DIAGNOSIS 2: Post-Weld Check ---
#     # If this number is '1', the lake and river have physically merged into one object.
#     print(f"DEBUG: Distinct islands after 100m buffer: {len(final_gdf)}")

    
#     # 4. FINAL FILTER: Keep only the islands that touch the river reference
#     # This deletes the lake even if it's huge, because it doesn't touch the line
#     river_ref_geom = gpd.read_file(river_ref).to_crs("EPSG:26912").union_all()
#     the_river_system = final_gdf[final_gdf.intersects(river_ref_geom)]

#     # --- DIAGNOSIS 3: Post-Filter Check ---
#     print(f"DEBUG: Islands remaining after River Reference filter: {len(the_river_system)}")
    
#     # 5. Shrink it back (undo the buffer) to get original river width
#     the_river_system['geometry'] = the_river_system.geometry.buffer(-100)
    
#     # 6. If multiple pieces still exist, take the largest of the survivors
#     if not the_river_system.empty:
#         the_river_system = the_river_system.loc[[the_river_system.geometry.area.idxmax()]]
        
#         final_poly = polygon_data / "Grand_Canyon_River_Master2.gpkg"
#         the_river_system.to_file(final_poly, driver="GPKG")
#         print("Master river system saved, lake removed.")

# #######################


DEBUG: Total input polygons from all tiles: 17
DEBUG: Distinct islands after 100m buffer: 2
Master river system saved, lake removed.


In [67]:
#### this scell works
file_list = [f for f in NDI_out.iterdir() if f.is_file()]
with rio.open(file_list[0]) as ref:
    target_crs = ref.crs
srcs = []
for f in file_list:
    src = rio.open(f)
    if src.crs != target_crs:
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        meta = src.meta.copy()
        meta.update({
            "crs": target_crs,
            "transform": transform,
            "width": width,
            "height": height
        })
    
        data = np.empty((src.count, height, width), dtype=src.dtypes[0])
    
        for i in range(1, src.count + 1):
            reproject(
                source=rio.band(src, i),
                destination=data[i-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=target_crs,
                resampling=Resampling.nearest
            )
        memfile = MemoryFile()
        dataset = memfile.open(**meta)
        dataset.write(data)
        srcs.append(dataset)
    else:
        srcs.append(src)
mosaic, transform = merge(srcs)

out_path = mosaic_out / "NDWI-TEST.tif"
meta = srcs[0].meta.copy()
meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": transform
})

with rio.open(out_path, "w", **meta) as dst:
    dst.write(mosaic)

print(f"Saved: {out_path}")

Saved: C:\RS_GIS\EGM722\Assignment\Grand_Canyon\USGS_data\Unzipped\Processed_data\Mosaic\NDWI-TEST.tif


In [14]:
def reproject_raster(src, dst_crs):
    transform, width, height = rio.warp.calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    data = np.empty((src.count, height, width), dtype=src.dtypes[0])

    for i in range(1, src.count + 1):
        reproject(
            source=rio.band(src, i),
            destination=data[i-1],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )

    return data, kwargs

In [58]:
def reproject_raster(src, dst_crs):
    transform, width, height = rio.warp.calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    data = np.empty((src.count, height, width), dtype=src.dtypes[0])

    for i in range(1, src.count + 1):
        reproject(
            source=rio.band(src, i),
            destination=data[i-1],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )

    return data, kwargs
    
mosaic_out = base/"Processed_data"/"Mosiac"
mosaic_out.mkdir(parents=True, exist_ok=True)

 
for yr in df["year"].unique():
    for colr in NDI:
        file_list = df[
            (df["year"] == yr) &
            (df["colour"] == colr)
            ]["path"].tolist()
        srcs = []
        for f in file_list:
            src = rio.open(f)
            if src.crs != target_crs:
                data, meta = reproject_raster(src, target_crs)
                srcs.append((data, meta))
            else:
                srcs.append((src.read(), src.meta))
        datasets = []
        for data, meta in srcs:
            memfile = MemoryFile()
            dataset = memfile.open(**meta)
            dataset.write(data)
            datasets.append(dataset)
        mosaic, transform = merge(datasets)
        print(mosaic.shape)
#            arrays.append(data)
#            srcs.append(src)
#            file_list = df[
#                (df["year"] == yr) &
#                (df["colour"] == colr)
#                ]["path"].tolist()
#            scrs = [rio.open(f) for f in file_list]
#            mosaic, transform = merge(srcs)
#            out_path = mosaic_out / f"{yr}_{colr}_mosaic.tif")
#            rio.merge.merge(file_list, dst_path = out_path)


(1, 11746, 19958)
(1, 11746, 19958)


In [55]:
def clipping(datafile):
    for file in parent.rglob(datafile):
        print(file)
        with rio.open(file) as data:
            geom = [feature["geometry"] for feature in boundary.__geo_interface__["features"]]
            out_image, out_transform = mask(data, geom, crop=True)
            fig, ax2 = plt.subplots(figsize=(10, 10))
            show(out_image, ax=ax2)